# Objetivo 2 — Implementación de Modelos HQCNN
**Modelos:** HQC-CNN · PEQML · HQCINN (shallow/deep)
**Framework:** PennyLane + PyTorch — `TorchLayer`
**Datasets:** etl_output/chest_xray · etl_output/lung_cancer

## 0 · Instalación de dependencias

In [1]:
import subprocess, sys
pkgs = [
    'torch torchvision --index-url https://download.pytorch.org/whl/cu118',
    'pennylane>=0.38',
    'pennylane-lightning[gpu]',
    'scikit-learn',
    'matplotlib seaborn tqdm',
]
for pkg in pkgs:
    subprocess.run([sys.executable,'-m','pip','install','-q']+pkg.split(), check=False)
print('OK')


OK


## 1 · Imports y seed global

In [2]:
import os, random, time
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torchvision.transforms as T
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, WeightedRandomSampler
import pennylane as qml
from pennylane.qnn import TorchLayer
from tqdm.notebook import tqdm
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix, roc_auc_score

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PennyLane:', qml.__version__)
print('PyTorch:  ', torch.__version__)
print('Device:   ', DEVICE)
if torch.cuda.is_available(): print('GPU:', torch.cuda.get_device_name(0))


PennyLane: 0.45.0
PyTorch:   2.12.0+cpu
Device:    cpu


## 2 · Rutas y DataLoaders

In [3]:
BASE_DIR  = Path(os.getcwd())
CHEST_OUT = BASE_DIR / 'etl_output' / 'chest_xray'
LUNG_OUT  = BASE_DIR / 'etl_output' / 'lung_cancer'
BATCH_SIZE = 32
IMG_SIZE   = (128, 128)
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

transform_train = T.Compose([
    T.Resize(IMG_SIZE), T.Grayscale(num_output_channels=3),
    T.RandomRotation(10), T.RandomHorizontalFlip(),
    T.ColorJitter(brightness=0.15),
    T.RandomAffine(degrees=0, scale=(0.90, 1.10)),
    T.ToTensor(), T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
transform_eval = T.Compose([
    T.Resize(IMG_SIZE), T.Grayscale(num_output_channels=3),
    T.ToTensor(), T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

def make_loaders(root, weighted=False):
    root = Path(root)
    ds_tr = ImageFolder(root/'train', transform=transform_train)
    ds_va = ImageFolder(root/'val',   transform=transform_eval)
    ds_te = ImageFolder(root/'test',  transform=transform_eval)
    if weighted:
        tgts = torch.tensor(ds_tr.targets)
        sw   = (1.0/torch.bincount(tgts).float())[tgts]
        loader_tr = DataLoader(ds_tr, batch_size=BATCH_SIZE,
                               sampler=WeightedRandomSampler(sw,len(sw),True),
                               num_workers=2, pin_memory=True)
    else:
        loader_tr = DataLoader(ds_tr, batch_size=BATCH_SIZE, shuffle=True,
                               num_workers=2, pin_memory=True)
    loader_va = DataLoader(ds_va, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
    loader_te = DataLoader(ds_te, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
    return loader_tr, loader_va, loader_te, ds_tr.class_to_idx

chest_train_loader, chest_val_loader, chest_test_loader, chest_classes = make_loaders(CHEST_OUT)
lung_train_loader,  lung_val_loader,  lung_test_loader,  lung_classes  = make_loaders(LUNG_OUT, weighted=True)
N_CLASSES_CHEST = len(chest_classes)  # 2
N_CLASSES_LUNG  = len(lung_classes)   # 3

ds_lung_tr = ImageFolder(LUNG_OUT/'train')
tgts = torch.tensor(ds_lung_tr.targets)
lung_cw = (1.0/torch.bincount(tgts).float())
lung_cw = (lung_cw/lung_cw.sum()).to(DEVICE)

print('Chest:', chest_classes, '| Lung:', lung_classes)
print('Lung class weights:', lung_cw)


Chest: {'NORMAL': 0, 'PNEUMONIA': 1} | Lung: {'Benign': 0, 'Malignant': 1, 'Normal': 2}
Lung class weights: tensor([0.6654, 0.1426, 0.1921])


## 3 · Backend PennyLane
`lightning.gpu` si CUDA, sino `default.qubit`.

In [4]:
def get_qdevice(n_qubits):
    if torch.cuda.is_available():
        try:
            dev = qml.device('lightning.gpu', wires=n_qubits)
            print(f'lightning.gpu ({n_qubits} qubits)')
            return dev
        except Exception:
            pass
    dev = qml.device('default.qubit', wires=n_qubits)
    print(f'default.qubit ({n_qubits} qubits)')
    return dev


## 4 · Modelo 1 — HQC-CNN (Dong et al., 2023)
CNN liviana + VQC 4 qubits, angle encoding RX, `StronglyEntanglingLayers`.

In [5]:
N_QUBITS = 4

dev_hqccnn = get_qdevice(N_QUBITS)

@qml.qnode(dev_hqccnn, interface='torch', diff_method='adjoint')
def circuit_hqccnn(inputs, weights):
    qml.AngleEmbedding(inputs, wires=range(N_QUBITS), rotation='X')
    qml.StronglyEntanglingLayers(weights, wires=range(N_QUBITS))
    return [qml.expval(qml.PauliZ(i)) for i in range(N_QUBITS)]

WS_HQCCNN = {'weights': (2, N_QUBITS, 3)}

class HQCCNN(nn.Module):
    def __init__(self, n_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3,16,3,padding=1), nn.BatchNorm2d(16), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16,32,3,padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32,64,3,padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.AdaptiveAvgPool2d((4,4)),
        )
        self.pre_q = nn.Sequential(
            nn.Linear(64*4*4, 64), nn.ReLU(),
            nn.Linear(64, N_QUBITS), nn.Tanh(),
        )
        self.q_layer    = TorchLayer(circuit_hqccnn, WS_HQCCNN)
        self.classifier = nn.Linear(N_QUBITS, n_classes)

    def forward(self, x):
        x = self.features(x).flatten(1)
        x = self.pre_q(x) * torch.pi
        x = self.q_layer(x)
        return self.classifier(x)

with torch.no_grad():
    m = HQCCNN(2)
    print('HQC-CNN output:', m(torch.randn(2,3,128,128)).shape)
    print('Params:', sum(p.numel() for p in m.parameters() if p.requires_grad))


default.qubit (4 qubits)
HQC-CNN output: torch.Size([2, 2])
Params: 89702


## 5 · Modelo 2 — PEQML (Abdur & Kim, 2025)
Backbone depthwise-separable + VQC 4 qubits, `BasicEntanglerLayers`.

In [6]:
dev_peqml = get_qdevice(N_QUBITS)

@qml.qnode(dev_peqml, interface='torch', diff_method='adjoint')
def circuit_peqml(inputs, weights):
    qml.AngleEmbedding(inputs, wires=range(N_QUBITS), rotation='Y')
    qml.BasicEntanglerLayers(weights, wires=range(N_QUBITS))
    return [qml.expval(qml.PauliZ(i)) for i in range(N_QUBITS)]

WS_PEQML = {'weights': (2, N_QUBITS)}

class PEQML(nn.Module):
    def __init__(self, n_classes):
        super().__init__()
        def dw(ic, oc, s=1):
            return nn.Sequential(
                nn.Conv2d(ic,ic,3,stride=s,padding=1,groups=ic,bias=False),
                nn.Conv2d(ic,oc,1,bias=False), nn.BatchNorm2d(oc), nn.ReLU6())
        self.features = nn.Sequential(
            nn.Conv2d(3,8,3,stride=2,padding=1,bias=False), nn.BatchNorm2d(8), nn.ReLU6(),
            dw(8,16,2), dw(16,32,2), dw(32,32,2),
            nn.AdaptiveAvgPool2d((2,2)),
        )
        self.pre_q = nn.Sequential(nn.Linear(32*2*2, N_QUBITS), nn.Tanh())
        self.q_layer    = TorchLayer(circuit_peqml, WS_PEQML)
        self.classifier = nn.Linear(N_QUBITS, n_classes)

    def forward(self, x):
        x = self.features(x).flatten(1)
        x = self.pre_q(x) * torch.pi
        x = self.q_layer(x)
        return self.classifier(x)

with torch.no_grad():
    m = PEQML(2)
    print('PEQML output:', m(torch.randn(2,3,128,128)).shape)
    print('Params:', sum(p.numel() for p in m.parameters() if p.requires_grad))


default.qubit (4 qubits)
PEQML output: torch.Size([2, 2])
Params: 3094


## 6 · Modelo 3 — HQCINN (Akpinar et al., 2025)
CNN estandar + VQC configurable: `shallow` (1 capa) o `deep` (3 capas), entrelazamiento `linear|circular|full`.

In [7]:
def build_hqcinn_qlayer(n_layers, entanglement='linear'):
    dev = get_qdevice(N_QUBITS)
    def entangle(wires, etype):
        n = len(wires)
        if etype == 'linear':
            for i in range(n-1): qml.CNOT(wires=[wires[i], wires[i+1]])
        elif etype == 'circular':
            for i in range(n):   qml.CNOT(wires=[wires[i], wires[(i+1)%n]])
        elif etype == 'full':
            for i in range(n):
                for j in range(i+1,n): qml.CNOT(wires=[wires[i], wires[j]])
    @qml.qnode(dev, interface='torch', diff_method='adjoint')
    def circuit(inputs, weights_rx, weights_ry, weights_rz):
        qml.AngleEmbedding(inputs, wires=range(N_QUBITS), rotation='X')
        for l in range(n_layers):
            for q in range(N_QUBITS):
                qml.RX(weights_rx[l,q], wires=q)
                qml.RY(weights_ry[l,q], wires=q)
                qml.RZ(weights_rz[l,q], wires=q)
            entangle(list(range(N_QUBITS)), entanglement)
        return [qml.expval(qml.PauliZ(i)) for i in range(N_QUBITS)]
    ws = {k:(n_layers, N_QUBITS) for k in ['weights_rx','weights_ry','weights_rz']}
    return TorchLayer(circuit, ws)

class HQCINN(nn.Module):
    VARIANTS = {'shallow':1, 'deep':3}
    def __init__(self, n_classes, variant='shallow', entanglement='linear'):
        super().__init__()
        n_layers = self.VARIANTS[variant]
        self.features = nn.Sequential(
            nn.Conv2d(3,32,3,padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32,64,3,padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64,128,3,padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.AdaptiveAvgPool2d((2,2)),
        )
        self.pre_q = nn.Sequential(nn.Linear(128*2*2, N_QUBITS), nn.Tanh())
        self.q_layer    = build_hqcinn_qlayer(n_layers, entanglement)
        self.classifier = nn.Linear(N_QUBITS, n_classes)
        self.variant = variant
    def forward(self, x):
        x = self.features(x).flatten(1)
        x = self.pre_q(x) * torch.pi
        x = self.q_layer(x)
        return self.classifier(x)

for v in ['shallow','deep']:
    with torch.no_grad():
        m = HQCINN(2, variant=v)
        out = m(torch.randn(2,3,128,128))
        p = sum(pp.numel() for pp in m.parameters() if pp.requires_grad)
        print(f'HQCINN-{v}: {out.shape}  params={p:,}')


default.qubit (4 qubits)
HQCINN-shallow: torch.Size([2, 2])  params=95,770
default.qubit (4 qubits)
HQCINN-deep: torch.Size([2, 2])  params=95,794


## 7 · Factory y tabla de parametros

In [8]:
def build_model(name, n_classes):
    if name == 'hqccnn':           return HQCCNN(n_classes)
    elif name == 'peqml':          return PEQML(n_classes)
    elif name == 'hqcinn_shallow': return HQCINN(n_classes, variant='shallow')
    elif name == 'hqcinn_deep':    return HQCINN(n_classes, variant='deep')
    else: raise ValueError(f'Desconocido: {name}')

print(f'{'Modelo':<20} {'n_classes=2':>14}  {'n_classes=3':>14}')
print('-'*52)
for mn in ['hqccnn','peqml','hqcinn_shallow','hqcinn_deep']:
    p2 = sum(p.numel() for p in build_model(mn,2).parameters() if p.requires_grad)
    p3 = sum(p.numel() for p in build_model(mn,3).parameters() if p.requires_grad)
    print(f'{mn:<20} {p2:>14,}  {p3:>14,}')


Modelo                  n_classes=2     n_classes=3
----------------------------------------------------
hqccnn                       89,702          89,707
peqml                         3,094           3,099
default.qubit (4 qubits)
default.qubit (4 qubits)
hqcinn_shallow               95,770          95,775
default.qubit (4 qubits)
default.qubit (4 qubits)
hqcinn_deep                  95,794          95,799
